In [1]:
import torch
from mmasim_kernels.amd.cdna3 import mfma_kernels

torch.manual_seed(0)
MFMA = mfma_kernels["f32_32x32x8_f16"]

In [2]:
bsz = 100
A = 10 * torch.randn(bsz, 128, 128, device='cuda:0', dtype=torch.float16)
B = 10 * torch.randn(bsz, 128, 128, device='cuda:0', dtype=torch.float16)
A, B

(tensor([[[ -9.2500,  -4.2539, -26.4375,  ...,  -2.1270,  -3.3164,  -2.0234],
          [-11.4531,  -5.7109,  -6.5078,  ...,  13.4219,  33.1562,  -8.4688],
          [  2.3750,  10.8906,  -2.4980,  ...,   2.4297,   1.7031,  -2.1113],
          ...,
          [  4.5703,  -5.8555,  -6.3438,  ...,  -4.8633,  -7.6172,   8.6016],
          [ -5.6602,   8.8359,   0.3926,  ...,   3.5664,   6.9844,  -0.8574],
          [  0.5459,  -1.7812,  -3.3223,  ...,  -1.3164,  -4.2109, -29.5469]],
 
         [[  1.3535,   2.5352,   9.0859,  ...,   6.6250, -11.8047,  -5.6094],
          [ -2.1406,  -2.1172,  16.6719,  ...,  -0.2306,   7.5039,  16.0625],
          [-15.6094,  -3.2188,   7.7266,  ...,  -3.6816,  12.8750,   2.1465],
          ...,
          [ -2.0605,  -4.3711,  -5.8750,  ...,  -4.1250,   1.0146,  -4.6016],
          [  9.9922,  -2.0137,  -0.8760,  ...,   6.2539,   6.7383,  -6.1133],
          [ 13.8203,  -5.0938,   5.4453,  ...,  -5.4492, -19.7031,   7.8750]],
 
         [[ -8.2188, -12.625

In [3]:
C_zeros = torch.zeros(bsz, 128, 128, device='cuda:0', dtype=torch.float32)
D_MFMA_accum = torch.zeros(bsz, 128, 128, device='cuda:0', dtype=torch.float32)
D_MFMA_no_accum = torch.zeros(bsz, 128, 128, device='cuda:0', dtype=torch.float32)
D_real = A.double() @ B.double()
for t in range(bsz):
    for i in range(0, 128, 32):
        for j in range(0, 128, 32):
            for k in range(0, 128, 8):
                D_MFMA_accum[t, i:i+32, j:j+32] = MFMA(A[t, i:i+32, k:k+8], B[t, k:k+8, j:j+32], D_MFMA_accum[t, i:i+32, j:j+32])
                D_MFMA_no_accum[t, i:i+32, j:j+32] += MFMA(A[t, i:i+32, k:k+8], B[t, k:k+8, j:j+32], C_zeros[t, i:i+32, j:j+32])

In [4]:
delta_accum = D_MFMA_accum - D_real
delta_no_accum = D_MFMA_no_accum - D_real
print(f"delta_accum mean:", delta_accum.mean().item())
print(f"delta_no_accum mean:", delta_no_accum.mean().item())

delta_accum mean: -1.2241043056580858e-06
delta_no_accum mean: -2.447148079384731e-08
